# 02 · Hallazgos medidos del mandato (Sección A y Anexo F.1)

Este cuaderno respalda, paso a paso, las cifras **[M] (medidas)** de la Sección A del Informe 1 y de la bitácora de supuestos (F.1). Cada paso dice **qué afirma el informe**, **cómo se calcula** y **qué resultado da**.

**Cómo leerlo**

- Cada sección empieza con un recuadro *Respalda:* que indica la parte del informe (A.3, A.4 o el n.º de supuesto de F.1).
- Los datos se leen con el mismo pipeline del repo: `etl/staging.py` (fuente tipada, sin limpiar) y `etl/processed.py` (limpia). Así las cifras salen de las mismas reglas que alimentan el modelo y el tablero.
- Al final hay una **tabla resumen** que compara cada cifra del informe con la que calcula este cuaderno.

**Cómo reproducir:** *Run All*, o `python notebooks/02_hallazgos_mandato.py` desde la raíz del repo (mismo código). No escribe archivos ni usa la API de tipo de cambio. Tarda segundos.

| Parte | Pasos | Respalda |
|---|---|---|
| 0. Carga | 0 | — |
| A. Período de análisis | 1–3 | A.3, supuestos 1, 2 y 3 |
| B. Moneda | 4–6 | A.3, supuestos 6 y 7 |
| C. Crecimiento en litros | 7–10 | A.4 (crecer), supuesto 1 |
| D. Descuentos e ingreso | 11–12 | A.3, supuesto 4 |
| E. Costos | 13–15 | A.3, supuesto 5 |
| F. Otros hallazgos de calidad | 16–18 | contexto para B y E |
| Resumen | — | todo lo anterior |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Raíz del repo: funciona desde notebooks/ (Jupyter) o desde la raíz (python notebooks/...).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "src"))  # por si el paquete no está instalado con pip install -e

from vina_vallehondo_bi.etl.staging import read_costs, read_dimensions, read_sales, read_shipments
from vina_vallehondo_bi.etl.processed import clean_costs, clean_markets, clean_sales, clean_shipments

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]


def pct(a, b) -> str:
    return f"{a / b * 100:.1f}%"


def mostrar(tabla) -> None:
    """En Jupyter muestra la tabla con formato y colores; en la terminal la imprime como texto."""
    if "ipykernel" in sys.modules:
        from IPython.display import display
        display(tabla)
    else:
        print(tabla.data if hasattr(tabla, "data") else tabla)
        print()


def colorear_variacion(variacion: pd.DataFrame):
    """Verde si el período creció contra el anterior, rojo si cayó."""
    def color(v):
        if pd.isna(v):
            return ""
        fondo = "#c8e6c9" if v > 0 else "#ffcdd2"
        return f"background-color: {fondo}; color: black"
    return variacion.style.map(color).format("{:+.1f}%", na_rep="")

## 0. Carga de datos

Se leen las 4 fuentes dos veces:

- **staging** (`stg_*`): la fuente tal como viene, solo con los tipos de dato corregidos. Sirve para **medir los problemas** de la fuente.
- **processed**: la fuente limpia con las reglas de `processed.py`. Sirve para **calcular las métricas**.

Después se arma `base`: una fila por venta limpia, con su país, canal, litros (cajas × litros por caja) y costo.

In [2]:
stg_ventas = read_sales(ROOT)
dims = read_dimensions(ROOT)
stg_costos = read_costs(ROOT)
stg_embarques = read_shipments(ROOT)

ventas = clean_sales(stg_ventas)
mercados = clean_markets(dims["stg_mercados"], dims["stg_zonas"], stg_ventas)
costos = clean_costs(stg_costos, dims["stg_cepas"])
embarques = clean_shipments(stg_embarques, ventas)

base = (ventas.merge(mercados[["mercado_id", "pais", "canal"]], on="mercado_id")
              .merge(costos[["producto_id", "anada", "tier", "litros_por_caja", "costo_caja_clp", "costo_imputado"]],
                     on=["producto_id", "anada"], how="left"))
base["litros"] = base.cajas * base.litros_por_caja
base["mes"] = base.fecha.dt.month
base["trimestre"] = "T" + base.fecha.dt.quarter.astype(str)

print(f"staging   · ventas: {len(stg_ventas):,} · costos: {len(stg_costos)} · líneas de embarque: {len(stg_embarques):,}")
print(f"processed · ventas: {len(ventas):,} · costos: {len(costos)} · líneas de embarque: {len(embarques):,}")

staging   · ventas: 13,968 · costos: 81 · líneas de embarque: 10,730
processed · ventas: 13,830 · costos: 96 · líneas de embarque: 10,715


# Parte A · Período de análisis

## 1. Ventas por año en la fuente

> **Respalda:** A.3 [M] *"Hay 13.885 ventas de 2025, 83 de 2026 y ninguna de 2024"* → el crecimiento se mide dentro de 2025 (supuesto 1).

Se cuentan las ventas de la fuente (sin limpiar) según el año de su fecha de factura.

In [3]:
por_anio = stg_ventas.fecha.dt.year.value_counts().reindex([2024, 2025, 2026], fill_value=0)
mostrar(por_anio.rename_axis("año").rename("ventas en la fuente").to_frame())

,ventas en la fuente
año,
2024,0
2025,13885
2026,83


**Resultado:** no hay ventas de 2024, así que no se puede comparar año contra año. Por eso el crecimiento se mide **entre meses (y trimestres) de 2025**.

## 2. Las 83 ventas de 2026

> **Respalda:** supuesto 3 *"Las 83 ventas de 2026 quedan fuera del análisis"*.

Se mira en qué meses caen y qué hace con ellas el pipeline.

In [4]:
v2026 = stg_ventas[stg_ventas.fecha.dt.year == 2026]
mostrar(v2026.fecha.dt.month.map(dict(enumerate(MESES, 1))).value_counts()
             .reindex(MESES[:6]).rename_axis("mes 2026").rename("ventas").to_frame())
print("venta_id repetidos dentro de esas 83:", int(v2026.venta_id.duplicated().sum()))
print("ventas con fecha reasignada a 2025 por el pipeline:", int(ventas.fecha_imputada.sum()))
imp = base[base.fecha_imputada]
print(f"peso de esas ventas: {imp.litros.sum():,.0f} litros = {pct(imp.litros.sum(), base.litros.sum())} de los litros 2025")

,ventas
mes 2026,
Ene,14
Feb,14
Mar,16
Abr,12
May,13
Jun,14


venta_id repetidos dentro de esas 83: 1
ventas con fecha reasignada a 2025 por el pipeline: 82
peso de esas ventas: 86,310 litros = 0.6% de los litros 2025


**Resultado:** son ventas sueltas de enero a junio de 2026 (un año incompleto).

⚠️ **Diferencia con el informe:** el supuesto 3 dice que *quedan fuera*, pero el pipeline (`clean_sales`, regla 5) **no las elimina**. Como `venta_id` es correlativo con la fecha, las trata como errores de digitación y les pone la fecha de la venta anterior, que es de 2025. Quedan 82 porque una es un `venta_id` repetido. Pesan menos del 1 % de los litros, así que el resultado casi no cambia, pero el texto del supuesto 3 y el pipeline deben decir lo mismo.

## 3. Por qué el período se asigna por fecha de factura

> **Respalda:** supuesto 2 *"El período de cada venta se asigna por su fecha de factura, no por el campo mes ni por la fecha de embarque"*.

Se comprueba que las otras dos opciones no son confiables:

- **campo `mes`** de la tabla de ventas: ¿coincide con el mes de la fecha?
- **fecha de embarque**: ¿cuántas ventas de exportación tienen una fecha de embarque creíble? (primera línea del embarque, mismo mercado de la venta y entre 0 y 60 días después).

In [5]:
distinto_mes = (stg_ventas.mes_origen != stg_ventas.fecha.dt.month).sum()
print(f"campo mes distinto del mes de la fecha: {distinto_mes} de {len(stg_ventas):,} ventas ({pct(distinto_mes, len(stg_ventas))})")

conf = embarques[embarques.fecha_embarque_confiable]
exp = base[base.canal == "Exportación"].merge(conf[["venta_id", "incoterm", "fecha_embarque"]], on="venta_id", how="left")
con_fecha = exp.fecha_embarque.notna()
print(f"ventas de exportación con fecha de embarque creíble: {int(con_fecha.sum()):,} de {len(exp):,} ({pct(con_fecha.sum(), len(exp))})")
print("de esas, embarcadas en 2026:", int((exp.fecha_embarque.dt.year == 2026).sum()))

campo mes distinto del mes de la fecha: 963 de 13,968 ventas (6.9%)
ventas de exportación con fecha de embarque creíble: 3,064 de 10,715 (28.6%)
de esas, embarcadas en 2026: 272


**Resultado:** el campo `mes` contradice la fecha en ~7 % de las ventas, y solo ~29 % de las exportaciones tiene una fecha de embarque creíble. La fecha de factura es la única completa y coincide con cómo cierra la viña [V].

# Parte B · Moneda

## 4. Ventas sin moneda

> **Respalda:** A.3 [M] *"695 (5 %) no tienen moneda"*.

Se cuentan las ventas de la fuente con `moneda_id` vacío, en total y por mercado.

In [6]:
stg_pais = stg_ventas.merge(mercados[["mercado_id", "pais"]], on="mercado_id")
sin_moneda = stg_pais.moneda_id.isna()
print(f"ventas sin moneda en la fuente: {sin_moneda.sum()} de {len(stg_ventas):,} ({pct(sin_moneda.sum(), len(stg_ventas))})")
mostrar(pd.DataFrame({
    "ventas": stg_pais.groupby("pais").size(),
    "sin_moneda": sin_moneda.groupby(stg_pais.pais).sum(),
}).assign(**{"sin_moneda_%": lambda t: t.sin_moneda / t.ventas * 100}))

ventas sin moneda en la fuente: 695 de 13,968 (5.0%)


,ventas,sin_moneda,sin_moneda_%
pais,,,
Alemania,1050,47,4.5
Brasil,1434,73,5.1
Canadá,1025,42,4.1
Chile,3152,136,4.3
China,1019,54,5.3
Corea del Sur,1095,73,6.7
Estados Unidos,1055,53,5.0
Japón,1065,52,4.9
México,1043,67,6.4


**Resultado:** el problema está repartido en todos los mercados (entre 4 % y 7 % en cada uno), no concentrado en uno.

## 5. Con qué moneda se completan

> **Respalda:** supuesto 6 *"Las ventas sin moneda se asumen en CLP si son de Chile y en USD si son de exportación"*, porque *"sus precios están en la misma escala que el resto de su mercado"* [M].

Se revisan dos cosas:

1. **Cada mercado factura en una sola moneda:** % de sus ventas con moneda que usan su moneda principal.
2. **Los precios de las ventas sin moneda tienen la escala de esa moneda:** precio mediano por caja con y sin moneda. Un precio en CLP es ~900 veces uno en USD, así que la escala delata la moneda.

In [7]:
monedas = dims["stg_monedas"].set_index("moneda_id").moneda
con = stg_pais[~sin_moneda]
principal = con.groupby("pais").moneda_id.agg(lambda s: s.mode().iloc[0])
mostrar(pd.DataFrame({
    "moneda_principal": principal.map(monedas),
    "%_ventas_en_esa_moneda": con.moneda_id.eq(con.pais.map(principal)).groupby(con.pais).mean() * 100,
    "precio_mediano_con_moneda": con.groupby("pais").precio_unitario.median(),
    "precio_mediano_sin_moneda": stg_pais[sin_moneda].groupby("pais").precio_unitario.median(),
}))
print("ventas con moneda completada por el pipeline:", int(ventas.moneda_imputada.sum()))
print("ventas sin moneda que eran filas duplicadas (se eliminan):", int(sin_moneda.sum() - ventas.moneda_imputada.sum()))

,moneda_principal,%_ventas_en_esa_moneda,precio_mediano_con_moneda,precio_mediano_sin_moneda
pais,,,,
Alemania,USD,100.0,43.0,43.0
Brasil,USD,100.0,43.1,43.3
Canadá,USD,100.0,43.2,33.5
Chile,CLP,100.0,"38,822.0","38,898.0"
China,USD,100.0,42.9,43.3
Corea del Sur,USD,100.0,43.2,41.9
Estados Unidos,USD,100.0,43.5,43.5
Japón,USD,100.0,43.0,42.4
México,USD,100.0,43.4,42.9


ventas con moneda completada por el pipeline: 691
ventas sin moneda que eran filas duplicadas (se eliminan): 4


**Resultado:** cada mercado factura el 100 % en una sola moneda (Chile en CLP, el resto en USD) y los precios sin moneda están en la misma escala. El pipeline completa 691 ventas; las otras 4 eran duplicados que se eliminan.

## 6. Monedas que existen en la fuente

> **Respalda:** supuesto 7 *"No hay ventas en euros"*: *"la tabla de monedas solo tiene CLP y USD"* [M].

In [8]:
mostrar(dims["stg_monedas"])

,moneda_id,moneda
0,1,CLP
1,2,USD


# Parte C · Crecimiento en litros

> **Respalda:** A.4 *"Crecer: variación porcentual de los litros netos vendidos entre cada mes del 2025, según fecha de factura. Un mercado creció si la variación es mayor que cero. Como la venta es estacional, también haremos comparaciones por trimestre"* y supuesto 1.

- **Litros** = cajas × litros por caja del producto (todos los productos son de 9 L, ver paso 15).
- **Netos**: las devoluciones (cajas negativas) restan litros.
- Período por **fecha de factura** (paso 3).

## 7. Litros por mes y mercado, 2025

Tabla de litros vendidos por mercado en cada mes. Se agregan:

- **Total 2025** y **Promedio mensual** de cada mercado (columnas de la derecha).
- **Total viña**: suma de todos los mercados en cada mes.
- **Promedio por mercado**: litros que vende en promedio un mercado en cada mes.

In [9]:
def tabla_litros(periodo: str, etiquetas: list) -> pd.DataFrame:
    """Litros por mercado y período, con totales y promedios."""
    t = base.pivot_table(index="pais", columns=periodo, values="litros", aggfunc="sum").astype(float)
    t.columns = etiquetas
    t.loc["Total viña"] = t.sum()
    t.loc["Promedio por mercado"] = t.drop(index="Total viña").mean()
    t["Total 2025"] = t[etiquetas].sum(axis=1)
    t[f"Promedio {'mensual' if periodo == 'mes' else 'trimestral'}"] = t[etiquetas].mean(axis=1)
    return t


litros_mes = tabla_litros("mes", MESES)
mostrar(litros_mes.style.format("{:,.0f}"))

,Ene,Feb,Mar,Abr,May,Jun,Jul,Ago,Sep,Oct,Nov,Dic,Total 2025,Promedio mensual
pais,,,,,,,,,,,,,,
Alemania,"125,793","86,661","110,349","87,948","94,779","76,617","83,943","102,780","75,771","84,321","76,374","90,189","1,095,525","91,294"
Brasil,"107,253","118,386","106,380","103,482","108,864","116,433","123,876","139,014","131,220","162,765","141,084","145,098","1,503,855","125,321"
Canadá,"105,750","75,411","101,673","97,119","88,425","67,077","91,440","80,730","95,598","94,248","80,802","89,442","1,067,715","88,976"
Chile,"330,048","263,196","273,807","299,493","260,190","251,514","277,317","274,554","242,784","292,338","262,116","279,504","3,306,861","275,572"
China,"97,362","76,797","95,769","72,063","68,382","91,053","73,278","88,074","102,204","94,977","74,331","112,275","1,046,565","87,214"
Corea del Sur,"126,576","91,179","107,046","110,133","83,106","97,344","90,072","94,986","96,921","84,825","79,695","96,642","1,158,525","96,544"
Estados Unidos,"111,735","104,301","106,812","78,579","108,621","81,558","99,846","85,473","73,170","72,846","107,775","85,185","1,115,901","92,992"
Japón,"109,782","101,925","109,368","109,296","113,535","98,595","75,276","82,017","90,297","77,130","89,244","67,653","1,124,118","93,676"
México,"114,966","85,086","92,088","83,790","88,083","79,686","79,362","86,139","88,785","103,824","92,853","101,781","1,096,443","91,370"


**Cómo leerla:** Chile es el mercado más grande (~3,3 millones de litros, 3 veces cualquier otro). Entre los de exportación, Brasil es el que más vende (~1,5 millones). El resto vende entre ~1,05 y 1,16 millones de litros en el año (~90 mil litros al mes).

## 8. Litros por trimestre y mercado, 2025

Lo mismo, agrupado por trimestre (T1 = ene–mar, T2 = abr–jun, T3 = jul–sep, T4 = oct–dic).

In [10]:
litros_tri = tabla_litros("trimestre", ["T1", "T2", "T3", "T4"])
mostrar(litros_tri.style.format("{:,.0f}"))

,T1,T2,T3,T4,Total 2025,Promedio trimestral
pais,,,,,,
Alemania,"322,803","259,344","262,494","250,884","1,095,525","273,881"
Brasil,"332,019","328,779","394,110","448,947","1,503,855","375,964"
Canadá,"282,834","252,621","267,768","264,492","1,067,715","266,929"
Chile,"867,051","811,197","794,655","833,958","3,306,861","826,715"
China,"269,928","231,498","263,556","281,583","1,046,565","261,641"
Corea del Sur,"324,801","290,583","281,979","261,162","1,158,525","289,631"
Estados Unidos,"322,848","268,758","258,489","265,806","1,115,901","278,975"
Japón,"321,075","321,426","247,590","234,027","1,124,118","281,030"
México,"292,140","251,559","254,286","298,458","1,096,443","274,111"


## 9. Meses de crecimiento por mercado

Variación de cada mes contra el mes anterior:

`variación = litros del mes / litros del mes anterior − 1`

🟩 **verde = creció** (variación > 0) · 🟥 **rojo = cayó**. Enero queda vacío porque no hay diciembre 2024 para comparar.

Columnas de resumen:

- **meses_al_alza**: cuántos de los 11 meses comparables crecieron.
- **tendencia_mensual_%**: pendiente de la recta ajustada a los 12 meses, como % del promedio mensual. Dice si el mercado **viene subiendo o bajando en el año**, sin que un mes puntual (un pedido grande) mueva la conclusión.

In [11]:
def tendencia(litros: pd.Series) -> float:
    """Pendiente de la recta litros ~ período, como % del promedio del período."""
    x = np.arange(1, len(litros) + 1)
    return np.polyfit(x, litros.values, 1)[0] / litros.mean() * 100


por_mes = litros_mes.drop(index="Promedio por mercado")[MESES]
var_mes = por_mes.pct_change(axis=1) * 100
mostrar(colorear_variacion(var_mes))

resumen_mes = pd.DataFrame({
    "meses_al_alza": (var_mes > 0).sum(axis=1),
    "tendencia_mensual_%": por_mes.apply(tendencia, axis=1),
}).sort_values("tendencia_mensual_%", ascending=False)
mostrar(resumen_mes.style.format({"tendencia_mensual_%": "{:+.1f}%"}))

,Ene,Feb,Mar,Abr,May,Jun,Jul,Ago,Sep,Oct,Nov,Dic
pais,,,,,,,,,,,,
Alemania,,-31.1%,+27.3%,-20.3%,+7.8%,-19.2%,+9.6%,+22.4%,-26.3%,+11.3%,-9.4%,+18.1%
Brasil,,+10.4%,-10.1%,-2.7%,+5.2%,+7.0%,+6.4%,+12.2%,-5.6%,+24.0%,-13.3%,+2.8%
Canadá,,-28.7%,+34.8%,-4.5%,-9.0%,-24.1%,+36.3%,-11.7%,+18.4%,-1.4%,-14.3%,+10.7%
Chile,,-20.3%,+4.0%,+9.4%,-13.1%,-3.3%,+10.3%,-1.0%,-11.6%,+20.4%,-10.3%,+6.6%
China,,-21.1%,+24.7%,-24.8%,-5.1%,+33.2%,-19.5%,+20.2%,+16.0%,-7.1%,-21.7%,+51.0%
Corea del Sur,,-28.0%,+17.4%,+2.9%,-24.5%,+17.1%,-7.5%,+5.5%,+2.0%,-12.5%,-6.0%,+21.3%
Estados Unidos,,-6.7%,+2.4%,-26.4%,+38.2%,-24.9%,+22.4%,-14.4%,-14.4%,-0.4%,+47.9%,-21.0%
Japón,,-7.2%,+7.3%,-0.1%,+3.9%,-13.2%,-23.7%,+9.0%,+10.1%,-14.6%,+15.7%,-24.2%
México,,-26.0%,+8.2%,-9.0%,+5.1%,-9.5%,-0.4%,+8.5%,+3.1%,+16.9%,-10.6%,+9.6%


,meses_al_alza,tendencia_mensual_%
pais,,
Brasil,7,+3.5%
China,5,+1.3%
Reino Unido,6,+0.1%
México,6,+0.1%
Total viña,5,-0.7%
Canadá,4,-0.7%
Chile,5,-0.8%
Países Bajos,4,-1.5%
Estados Unidos,4,-2.2%


**Cómo leerla:** mes a mes los litros suben y bajan mucho (±20–50 %), en parte porque un pedido grande mueve todo un mes y porque febrero es más corto. Por eso la variación de un mes aislado no basta para decir que un mercado *creció*: hay que mirar cuántos meses crece y su tendencia. **Brasil** es el único que crece de forma sostenida (+3,5 % por mes y 7 de 11 meses al alza); **China** le sigue (+1,3 %). **Japón** (−3,8 %), **Alemania**, **Corea del Sur** y **Estados Unidos** vienen cayendo. La viña en total cae levemente.

### 9.b Cada mes contra el promedio 2025 del mercado

Otra forma de leer el crecimiento: comparar cada mes con el **promedio mensual 2025 de ese mismo mercado** (columna *Promedio mensual* del paso 7), en vez de con el mes anterior.

`variación vs promedio = litros del mes / promedio mensual 2025 del mercado − 1`

🟩 **verde = mes sobre su promedio** · 🟥 **rojo = bajo su promedio**. Así un mes no depende de si el anterior fue bueno o malo. Si un mercado crece en el año, sus meses rojos se juntan al inicio y los verdes al final; si cae, al revés.

- **meses_sobre_promedio**: cuántos de los 12 meses superan el promedio.
- **sobre_promedio_ene_jun** / **sobre_promedio_jul_dic**: cuántos de ellos caen en cada mitad del año.

In [12]:
vs_prom_mes = por_mes.div(por_mes.mean(axis=1), axis=0) * 100 - 100
mostrar(colorear_variacion(vs_prom_mes))

sobre = vs_prom_mes > 0
mostrar(pd.DataFrame({
    "meses_sobre_promedio": sobre.sum(axis=1),
    "sobre_promedio_ene_jun": sobre[MESES[:6]].sum(axis=1),
    "sobre_promedio_jul_dic": sobre[MESES[6:]].sum(axis=1),
}).sort_values("sobre_promedio_jul_dic", ascending=False))

,Ene,Feb,Mar,Abr,May,Jun,Jul,Ago,Sep,Oct,Nov,Dic
pais,,,,,,,,,,,,
Alemania,+37.8%,-5.1%,+20.9%,-3.7%,+3.8%,-16.1%,-8.1%,+12.6%,-17.0%,-7.6%,-16.3%,-1.2%
Brasil,-14.4%,-5.5%,-15.1%,-17.4%,-13.1%,-7.1%,-1.2%,+10.9%,+4.7%,+29.9%,+12.6%,+15.8%
Canadá,+18.9%,-15.2%,+14.3%,+9.2%,-0.6%,-24.6%,+2.8%,-9.3%,+7.4%,+5.9%,-9.2%,+0.5%
Chile,+19.8%,-4.5%,-0.6%,+8.7%,-5.6%,-8.7%,+0.6%,-0.4%,-11.9%,+6.1%,-4.9%,+1.4%
China,+11.6%,-11.9%,+9.8%,-17.4%,-21.6%,+4.4%,-16.0%,+1.0%,+17.2%,+8.9%,-14.8%,+28.7%
Corea del Sur,+31.1%,-5.6%,+10.9%,+14.1%,-13.9%,+0.8%,-6.7%,-1.6%,+0.4%,-12.1%,-17.5%,+0.1%
Estados Unidos,+20.2%,+12.2%,+14.9%,-15.5%,+16.8%,-12.3%,+7.4%,-8.1%,-21.3%,-21.7%,+15.9%,-8.4%
Japón,+17.2%,+8.8%,+16.8%,+16.7%,+21.2%,+5.3%,-19.6%,-12.4%,-3.6%,-17.7%,-4.7%,-27.8%
México,+25.8%,-6.9%,+0.8%,-8.3%,-3.6%,-12.8%,-13.1%,-5.7%,-2.8%,+13.6%,+1.6%,+11.4%


,meses_sobre_promedio,sobre_promedio_ene_jun,sobre_promedio_jul_dic
pais,,,
Brasil,5,0,5
Canadá,7,3,4
Reino Unido,7,3,4
China,7,3,4
México,5,2,3
Chile,5,2,3
Estados Unidos,6,4,2
Corea del Sur,6,4,2
Alemania,4,3,1


**Cómo leerla:** **Brasil** pasa los 6 primeros meses bajo su promedio y de agosto a diciembre los 5 sobre él (entre +5 % y +30 %): creció durante el año. **Japón** es el caso opuesto: sus 6 meses sobre el promedio son ene–jun y todo jul–dic queda bajo (hasta −28 % en diciembre). **Países Bajos**, **Alemania**, **Corea del Sur** y **Estados Unidos** también concentran sus meses buenos en la primera mitad. Los demás alternan verde y rojo sin un patrón claro, coherente con su tendencia cercana a cero.

## 10. Trimestres de crecimiento por mercado

Lo mismo por trimestre (cada trimestre contra el anterior). Al juntar 3 meses, el ruido de los pedidos grandes se compensa y la estacionalidad se ve mejor.

In [13]:
por_tri = litros_tri.drop(index="Promedio por mercado")[["T1", "T2", "T3", "T4"]]
var_tri = por_tri.pct_change(axis=1) * 100
var_tri["T4 vs T1"] = (por_tri.T4 / por_tri.T1 - 1) * 100
var_tri["trimestres_al_alza"] = (var_tri[["T2", "T3", "T4"]] > 0).sum(axis=1)
var_tri = var_tri.sort_values("T4 vs T1", ascending=False)
mostrar(colorear_variacion(var_tri.drop(columns="trimestres_al_alza"))
        .relabel_index(["T1", "T2 vs T1", "T3 vs T2", "T4 vs T3", "T4 vs T1"], axis=1))
mostrar(var_tri[["trimestres_al_alza"]])

,T1,T2 vs T1,T3 vs T2,T4 vs T3,T4 vs T1
pais,,,,,
Brasil,,-1.0%,+19.9%,+13.9%,+35.2%
Reino Unido,,+16.2%,-4.5%,-5.0%,+5.4%
China,,-14.2%,+13.8%,+6.8%,+4.3%
México,,-13.9%,+1.1%,+17.4%,+2.2%
Chile,,-6.4%,-2.0%,+4.9%,-3.8%
Total viña,,-7.3%,-1.3%,+3.5%,-5.3%
Canadá,,-10.7%,+6.0%,-1.2%,-6.5%
Países Bajos,,-1.5%,-15.5%,+10.1%,-8.4%
Estados Unidos,,-16.8%,-3.8%,+2.8%,-17.7%


,trimestres_al_alza
pais,
Brasil,2
Reino Unido,1
China,2
México,2
Chile,1
Total viña,1
Canadá,1
Países Bajos,1
Estados Unidos,1


**Cómo leerla:** se confirma lo de los meses. **Brasil** crece en T3 y T4 y termina el año +35 % sobre su T1, muy por encima del resto; **Reino Unido**, **China** y **México** terminan levemente arriba (+2 % a +5 %). **Japón**, **Alemania**, **Corea del Sur** y **Estados Unidos** terminan entre −18 % y −27 % bajo su T1.

### 10.b Cada trimestre contra el promedio 2025 del mercado

Lo mismo que el paso 9.b, por trimestre: `litros del trimestre / promedio trimestral 2025 del mercado − 1` (columna *Promedio trimestral* del paso 8).

In [14]:
vs_prom_tri = por_tri.div(por_tri.mean(axis=1), axis=0) * 100 - 100
vs_prom_tri = vs_prom_tri.sort_values("T4", ascending=False)
mostrar(colorear_variacion(vs_prom_tri))

,T1,T2,T3,T4
pais,,,,
Brasil,-11.7%,-12.6%,+4.8%,+19.4%
México,+6.6%,-8.2%,-7.2%,+8.9%
China,+3.2%,-11.5%,+0.7%,+7.6%
Chile,+4.9%,-1.9%,-3.9%,+0.9%
Total viña,+5.6%,-2.1%,-3.4%,-0.0%
Canadá,+6.0%,-5.4%,+0.3%,-0.9%
Países Bajos,+7.1%,+5.5%,-10.8%,-1.8%
Reino Unido,-7.5%,+7.5%,+2.6%,-2.5%
Estados Unidos,+15.7%,-3.7%,-7.3%,-4.7%


**Cómo leerla:** **Brasil** pasa de ~12 % bajo su promedio en T1–T2 a +5 % en T3 y +19 % en T4. **Japón** hace lo contrario: +14 % en T1–T2 y −12 % / −17 % en T3–T4. **Alemania**, **Corea del Sur** y **Estados Unidos** tienen su mejor trimestre en T1 (+12 % a +18 %) y terminan el año bajo su promedio.

# Parte D · Descuentos e ingreso

## 11. Descuento promedio por mercado

> **Respalda:** A.3 [M] *"Brasil tiene un descuento promedio de 13,8 %, frente a 2,2 %–3,2 % en los otros 10 mercados"*.

Dos formas de promediar (solo ventas, sin devoluciones):

- **descuento_simple_%**: promedio del descuento de cada venta (todas pesan igual).
- **descuento_ponderado_%**: Σ(cajas × precio × descuento) / Σ(cajas × precio). Las ventas grandes pesan más; es el % del monto bruto que efectivamente se descuenta.

In [15]:
pos = base[base.cajas > 0]
bruto = pos.cajas * pos.precio_unitario
descuentos = pd.DataFrame({
    "descuento_simple_%": pos.groupby("pais").descuento.mean() * 100,
    "descuento_ponderado_%": (bruto * pos.descuento).groupby(pos.pais).sum() / bruto.groupby(pos.pais).sum() * 100,
    "descuento_min_%": pos.groupby("pais").descuento.min() * 100,
    "descuento_max_%": pos.groupby("pais").descuento.max() * 100,
}).sort_values("descuento_simple_%", ascending=False)
mostrar(descuentos)
otros = descuentos.drop(index="Brasil")["descuento_simple_%"]
print(f"otros 10 mercados: entre {otros.min():.1f}% y {otros.max():.1f}%")

,descuento_simple_%,descuento_ponderado_%,descuento_min_%,descuento_max_%
pais,,,,
Brasil,13.8,13.8,10.0,18.0
China,2.8,2.9,0.0,8.0
Corea del Sur,2.7,2.8,0.0,8.0
Alemania,2.7,2.6,0.0,8.0
Reino Unido,2.7,2.6,0.0,8.0
Japón,2.6,2.6,0.0,8.0
México,2.6,2.7,0.0,8.0
Chile,2.6,2.7,0.0,8.0
Países Bajos,2.6,2.5,0.0,8.0


otros 10 mercados: entre 2.6% y 2.8%


Para ver si el descuento de Brasil es una política del mercado o un cliente puntual, se separa por cliente:

In [16]:
mostrar((pos[pos.pais == "Brasil"].groupby("cliente_id").descuento.mean() * 100).rename("descuento promedio %").to_frame())

,descuento promedio %
cliente_id,
4,13.9
5,13.6
6,13.8


**Resultado:** Brasil descuenta ~13,8 % y lo hace parejo en sus 3 clientes (es una política del mercado, coherente con el descuento especial "para entrar con más fuerza" [V]). Los otros 10 mercados promedian entre 2,6 % y 2,8 %.

⚠️ **Diferencia con el informe:** A.3 dice *"2,2 %–3,2 %"* para los otros mercados; con estos datos el rango es **2,6 %–2,8 %**. Además, el acta (F.3) menciona a **Países Bajos** con ~13 % de descuento; según los datos es **Brasil**, Países Bajos tiene 2,6 %.

## 12. Ingreso antes y después de descuentos

> **Respalda:** supuesto 4 *"El ingreso se mide después de descuentos (monto neto facturado)"*: medido antes, no mostraría lo que la viña recibe.

Se compara el monto bruto (cajas × precio) con el neto (después del descuento) en la moneda de cada mercado.

In [17]:
ingreso = pd.DataFrame({
    "moneda": pos.groupby("pais").moneda_id.first().map(monedas),
    "monto_bruto": bruto.groupby(pos.pais).sum(),
    "monto_neto": pos.groupby("pais").monto_neto.sum(),
})
ingreso["se_pierde_por_descuento_%"] = (1 - ingreso.monto_neto / ingreso.monto_bruto) * 100
mostrar(ingreso.sort_values("se_pierde_por_descuento_%", ascending=False).style.format(
    {"monto_bruto": "{:,.0f}", "monto_neto": "{:,.0f}", "se_pierde_por_descuento_%": "{:.1f}%"}))

,moneda,monto_bruto,monto_neto,se_pierde_por_descuento_%
pais,,,,
Brasil,USD,"7,473,516","6,445,566",13.8%
China,USD,"5,211,670","5,060,987",2.9%
Corea del Sur,USD,"5,862,680","5,697,072",2.8%
México,USD,"5,643,223","5,492,690",2.7%
Chile,CLP,"15,286,101,465","14,880,291,082",2.7%
Alemania,USD,"5,500,325","5,355,230",2.6%
Reino Unido,USD,"5,625,407","5,478,030",2.6%
Japón,USD,"5,740,180","5,590,989",2.6%
Estados Unidos,USD,"5,765,791","5,617,728",2.6%


**Resultado:** el descuento se lleva ~14 % del monto bruto de Brasil y entre 2,5 % y 2,9 % en el resto. Como el descuento no es parejo entre mercados, usar el monto bruto cambiaría la comparación entre ellos.

# Parte E · Costos

## 13. Ventas sin costo para su producto y añada

> **Respalda:** A.3 [M] *"17 % de las ventas no tiene costo para su producto y añada"*.

El costo viene en `origen_costos_productos.csv`, con una fila por producto y añada. Se cruza cada venta de la fuente con ese archivo por (`producto_id`, `anada`) y se cuentan las que no encuentran costo. Hay dos motivos:

- **sin fila**: la combinación producto-añada no existe en el archivo.
- **costo vacío**: la fila existe, pero el costo viene vacío.

In [18]:
cruce = stg_ventas.merge(stg_costos[["producto_id", "anada", "costo_caja_clp"]],
                         on=["producto_id", "anada"], how="left", indicator=True)
sin_fila = cruce._merge == "left_only"
costo_vacio = (cruce._merge == "both") & cruce.costo_caja_clp.isna()
sin_costo = sin_fila | costo_vacio
print(f"ventas sin costo para su producto y añada: {int(sin_costo.sum()):,} de {len(cruce):,} ({pct(sin_costo.sum(), len(cruce))})")
print(f"  · sin fila en el archivo de costos: {int(sin_fila.sum()):,}")
print(f"  · fila con costo vacío:             {int(costo_vacio.sum()):,}")

detalle = (cruce[sin_costo].assign(motivo=np.where(sin_fila[sin_costo], "sin fila", "costo vacío"))
           .groupby(["producto_id", "anada", "motivo"]).size().rename("ventas").reset_index())
mostrar(detalle)

ventas sin costo para su producto y añada: 2,378 de 13,968 (17.0%)
  · sin fila en el archivo de costos: 2,194
  · fila con costo vacío:             184


,producto_id,anada,motivo,ventas
0,2,2021,sin fila,187
1,2,2024,costo vacío,184
2,6,2021,sin fila,85
3,8,2021,sin fila,156
4,10,2021,sin fila,197
5,13,2021,sin fila,186
6,14,2021,sin fila,180
7,16,2021,sin fila,172
8,17,2021,sin fila,191
9,18,2021,sin fila,104


**Resultado:** 17,0 % de las ventas (2.378) no tiene costo propio: 15 productos no tienen fila para la añada 2021 y el producto 2 tiene el costo vacío en la añada 2024 (16 combinaciones producto-añada en total).

## 14. Cómo se completa el costo faltante

> **Respalda:** que el 17 % se *"resuelve con supuestos declarados (F.1)"*.

Se comprueba que el costo por caja **depende solo del tier (nivel de calidad) y la añada**: todos los productos del mismo tier y añada tienen el mismo costo. Si es así, el costo faltante se puede completar con el de su tier y añada sin inventar nada.

In [19]:
con_costo = stg_costos.dropna(subset=["costo_caja_clp"])
variantes = con_costo.groupby(["tier", "anada"]).costo_caja_clp.nunique()
print("combinaciones tier-añada con más de un costo distinto:", int((variantes > 1).sum()))
print()
print("Costo por caja (CLP) según tier y añada:")
mostrar(con_costo.pivot_table(index="tier", columns="anada", values="costo_caja_clp", aggfunc="first")
                 .style.format("{:,.0f}"))
print(f"combinaciones producto-añada completadas por el pipeline: {int(costos.costo_imputado.sum())} de {len(costos)}")
print(f"ventas limpias que usan un costo completado: {int(base.costo_imputado.sum()):,} ({pct(base.costo_imputado.sum(), len(base))})")
print("ventas limpias que siguen sin costo:", int(base.costo_caja_clp.isna().sum()))

combinaciones tier-añada con más de un costo distinto: 0

Costo por caja (CLP) según tier y añada:


anada,2021,2022,2023,2024
tier,,,,
Gran Reserva,"35,316","34,344","33,372","32,400"
Reserva,"21,582","20,988","20,394","19,800"
Varietal,"13,734","13,356","12,978","12,600"


combinaciones producto-añada completadas por el pipeline: 16 de 96
ventas limpias que usan un costo completado: 2,351 (17.0%)
ventas limpias que siguen sin costo: 0


**Resultado:** ningún tier-añada tiene dos costos distintos, así que completar con el costo del tier es exacto. Después de limpiar, **ninguna venta queda sin costo** (17 % de ellas usa un costo completado).

⚠️ **Falta en F.1:** el informe no tiene un supuesto que declare esta regla. Sugerencia: *"El costo faltante de un producto y añada se completa con el de su tier y añada, porque el costo depende solo de esas dos variables [M]"*.

## 15. Qué costos traen las fuentes

> **Respalda:** supuesto 5 *"El margen descuenta solo el costo de producto; flete, seguro, aranceles, transporte, mano de obra, entre otros quedan fuera"*, porque *"no están en las 4 fuentes"*.

Se listan las columnas del archivo de costos y los litros por caja de cada formato.

In [20]:
print("columnas del archivo de costos:", list(stg_costos.columns))
print("litros por caja por formato:", {k: sorted(map(float, s)) for k, s in stg_costos.groupby("formato").litros_por_caja.unique().items()})

columnas del archivo de costos: ['fila_origen', 'producto_id', 'producto', 'cepa', 'tier', 'formato', 'litros_por_caja', 'anada', 'costo_caja_clp', 'activo']
litros por caja por formato: {'12x750': [9.0], '6x1500': [9.0]}


**Resultado:** el único costo es `costo_caja_clp` (costo de producto por caja); no hay columnas de flete, seguro ni aranceles. Ambos formatos (12×750 ml y 6×1,5 L) son de 9 litros por caja, por eso litros = cajas × 9.

# Parte F · Otros hallazgos de calidad

No se citan como cifra en la Sección A, pero explican reglas de limpieza que usan las Secciones B (cadena de datos) y E (cobertura).

## 16. Correcciones de cajas, montos y duplicados

In [21]:
print("venta_id repetidos en la fuente (se conserva el primero):", int(stg_ventas.venta_id.duplicated().sum()))
print("ajuste de cajas:", ventas.ajuste_cajas.value_counts().to_dict())
print(f"monto recalculado (cajas × precio × (1 − descuento)): {int(ventas.monto_recalculado.sum())} ({pct(ventas.monto_recalculado.sum(), len(ventas))})")
dev = base[base.es_devolucion]
print(f"devoluciones (cajas negativas): {len(dev)} ventas · {dev.litros.sum():,.0f} litros "
      f"({pct(-dev.litros.sum(), base.litros.sum())} de los litros 2025)")

venta_id repetidos en la fuente (se conserva el primero): 138
ajuste de cajas: {'ninguno': 13637, 'botellas_a_cajas': 138, 'dividido_por_10': 55}
monto recalculado (cajas × precio × (1 − descuento)): 691 (5.0%)
devoluciones (cajas negativas): 110 ventas · -125,973 litros (0.9% de los litros 2025)


## 17. Embarques y proporción de litros en CIF

En **CIF** la viña paga el flete y el seguro hasta el puerto de destino; en **FOB** los paga el comprador. Como el flete no está en las fuentes (paso 15), el margen de las ventas CIF queda sobrestimado. Si la proporción CIF es parecida entre mercados, ese sesgo es parejo y no cambia el orden entre ellos.

Se calcula sobre las ventas con embarque creíble (paso 3).

In [22]:
print(f"líneas de embarque: {len(stg_embarques):,} · sin venta (descartadas): {len(stg_embarques) - len(embarques)}")
print("embarque_id repetidos en la fuente:", int(stg_embarques.drop_duplicates("embarque_orden").embarque_id.duplicated().sum()))
cif = (exp[exp.incoterm == "CIF"].groupby("pais").litros.sum()
       / exp[exp.incoterm.notna()].groupby("pais").litros.sum() * 100)
mostrar(cif.astype(float).sort_values().rename("% litros en CIF (embarques creíbles)").to_frame())

líneas de embarque: 10,730 · sin venta (descartadas): 15
embarque_id repetidos en la fuente: 15


,% litros en CIF (embarques creíbles)
pais,
Estados Unidos,36.1
Brasil,36.2
Alemania,36.5
Reino Unido,38.4
China,38.5
Corea del Sur,39.9
México,40.6
Países Bajos,40.8
Canadá,41.2


## 18. Tablas maestras

In [23]:
zonas = dims["stg_zonas"]
huerfanos = dims["stg_mercados"][~dims["stg_mercados"].zona_id.isin(zonas.zona_id)]
print("mercados con zona_id inexistente en la fuente:", huerfanos[["pais", "zona_id"]].to_dict("records"))
print("zona asignada por el pipeline:", mercados.loc[mercados.mercado_id.isin(huerfanos.mercado_id), ["pais", "zona"]].to_dict("records"))

mercados con zona_id inexistente en la fuente: [{'pais': 'Corea del Sur', 'zona_id': 8}]
zona asignada por el pipeline: [{'pais': 'Corea del Sur', 'zona': 'Asia'}]


# Resumen: cifras del informe vs. este cuaderno

Cada fila es una cifra [M] del Informe 1. **Coincide** = el cuaderno respalda la cifra tal como está escrita; **Revisar** = hay que corregir el texto del informe.

In [24]:
resumen = pd.DataFrame([
    ("A.3", "Ventas 2025 / 2026 / 2024", "13.885 / 83 / 0",
     f"{por_anio[2025]:,} / {por_anio[2026]} / {por_anio[2024]}".replace(",", "."), "Coincide", "paso 1"),
    ("A.3", "Descuento promedio Brasil", "13,8 %",
     f"{descuentos.loc['Brasil', 'descuento_simple_%']:.1f} %".replace(".", ","), "Coincide", "paso 11"),
    ("A.3", "Descuento otros 10 mercados", "2,2 %–3,2 %",
     f"{otros.min():.1f} %–{otros.max():.1f} %".replace(".", ","), "Revisar", "paso 11"),
    ("A.3", "Ventas sin costo para su producto y añada", "17 %",
     pct(sin_costo.sum(), len(cruce)).replace(".", ","), "Coincide", "paso 13"),
    ("A.3 / S6", "Ventas sin moneda", "695 (5 %)",
     f"{sin_moneda.sum()} ({pct(sin_moneda.sum(), len(stg_ventas))})".replace(".", ","), "Coincide", "paso 4"),
    ("S3", "Tratamiento de las ventas 2026", "quedan fuera",
     f"{int(ventas.fecha_imputada.sum())} reasignadas a 2025", "Revisar", "paso 2"),
    ("S6", "Moneda de las ventas sin moneda", "CLP Chile, USD exportación",
     "100 % de cada mercado en una moneda", "Coincide", "paso 5"),
    ("S7", "Monedas en la fuente", "CLP y USD", " y ".join(dims["stg_monedas"].moneda), "Coincide", "paso 6"),
    ("S5", "Costos en las fuentes", "solo costo de producto", "solo costo_caja_clp", "Coincide", "paso 15"),
    ("F.3", "Mercado con descuento ~13 %", "Países Bajos", "Brasil", "Revisar", "paso 11"),
    ("F.1", "Regla para completar costos", "(no está)", "costo del tier y añada", "Agregar", "paso 14"),
], columns=["sección", "cifra", "informe", "cuaderno", "estado", "dónde"])
mostrar(resumen.set_index(["sección", "cifra"]))

informe                             cuaderno    estado    dónde
sección  cifra                                                                                                                        
A.3      Ventas 2025 / 2026 / 2024                             13.885 / 83 / 0                      13.885 / 83 / 0  Coincide   paso 1
         Descuento promedio Brasil                                      13,8 %                               13,8 %  Coincide  paso 11
         Descuento otros 10 mercados                               2,2 %–3,2 %                          2,6 %–2,8 %   Revisar  paso 11
         Ventas sin costo para su producto y añada                        17 %                                17,0%  Coincide  paso 13
A.3 / S6 Ventas sin moneda                                           695 (5 %)                           695 (5,0%)  Coincide   paso 4
S3       Tratamiento de las ventas 2026                           quedan fuera                82 reasignadas a 2025   Revisar   paso 2
S6       Moneda de las ventas sin moneda            CLP Chile, USD exportación  100 % de cada mercado en una moneda  Coincide   paso 5
S7       Monedas en la fuente                                        CLP y USD                            CLP y USD  Coincide   paso 6
S5       Costos en las fuentes                          solo costo de producto                  solo costo_caja_clp  Coincide  paso 15
F.3      Mercado con descuento ~13 %                              Países Bajos                               Brasil   Revisar  paso 11
F.1      Regla para completar costos                                 (no está)               costo del tier y añada   Agregar  paso 14

**No se calcula aquí:** el margen por litro y la regla de *buen margen* (A.4, supuesto 8). Necesitan el dólar observado diario (API de Mindicador) y son parte de la ficha de la métrica (Sección D).